# 02 — Cleaning & ETL

**Airline Operations Intelligence Platform** · Notebook 2 of 10 · *runs locally*

## Purpose
Implement the 11 rules that notebook `01` derived from evidence, and produce the single
curated dataset every downstream notebook reads.

## Design principle: fail loudly, never silently
The defect notebook 01 found — a join that silently deletes 486,165 rows — is the
canonical big-data failure: **the pipeline succeeds and the numbers are wrong.**
So every stage here asserts a row-count contract. If a transformation loses rows it
was not supposed to lose, this notebook raises instead of writing bad data.

## Rules implemented
| # | Rule | Section |
|---|---|---|
| 1 | Recover October's DOT airport codes → IATA | §3 |
| 5 | Deduplicate on the business key | §4 |
| 2, 3, 4, 11 | Preserve structural nulls; classify flight outcome | §5 |
| 7 | `HHMM` integers → minutes + real timestamps | §6 |
| 6 | Keep negative delays (early departures) | §7 |
| 8 | Derive the ML target and analysis dimensions | §7 |
| 9, 10 | Enrich with airline and airport metadata | §8 |

## Output
`data/curated/flights.parquet` — partitioned by month, read by notebooks 03–09.

---
## 1. Setup

In [ ]:
import sys, time
sys.path.insert(0, "../src")

from config import build_spark, PATHS
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = build_spark("02-etl")

flights_raw  = spark.read.parquet(str(PATHS["raw"] / "flights_raw.parquet"))
airlines_raw = spark.read.parquet(str(PATHS["raw"] / "airlines_raw.parquet"))
airports_raw = spark.read.parquet(str(PATHS["raw"] / "airports_raw.parquet"))

N_RAW = flights_raw.count()
print(f"Raw flights : {N_RAW:,}")
print(f"Airlines    : {airlines_raw.count()}")
print(f"Airports    : {airports_raw.count()}")

---
## 2. The row-count contract

A tiny helper that makes data loss impossible to miss. Every stage declares how many
rows it expects to keep; a mismatch raises immediately.

In [ ]:
class Contract:
    """Tracks row counts through the pipeline and refuses silent loss."""

    def __init__(self, start: int, label: str = "raw"):
        self.n = start
        self.log = [(label, start, 0)]
        print(f"{'STAGE':<34}{'ROWS':>12}{'DELTA':>12}")
        print("-" * 58)
        print(f"{label:<34}{start:>12,}{'':>12}")

    def check(self, df, label: str, expect_delta: int = 0, tolerance: int = 0):
        n = df.count()
        delta = n - self.n
        print(f"{label:<34}{n:>12,}{delta:>+12,}")
        if abs(delta - expect_delta) > tolerance:
            raise AssertionError(
                f"{label}: expected delta {expect_delta:+,} (tol {tolerance}), got {delta:+,}"
            )
        self.n = n
        self.log.append((label, n, delta))
        return df

contract = Contract(N_RAW)

---
## 3. Rule 1 — recover October's airport codes

Notebook 01 established that October (486,165 flights, 8.4%) stores airport identifiers as
5-digit DOT codes while every other month uses 3-letter IATA. `airports.csv` is IATA-keyed,
so those rows cannot be joined and would vanish.

**No external lookup table is used.** The mapping is recovered from the dataset itself:

1. `DISTANCE` is a deterministic function of the (origin, destination) pair, so the *set of
   distances flown out of an airport* is a strong fingerprint.
2. Fingerprints alone tie for small airports (23 different DOT codes match ATL's set
   perfectly), so flight **volume** is added as a second signal — October volume should be
   roughly one eleventh of the airport's volume across the other 11 months.
3. The combined score is resolved to a strict **one-to-one assignment**, since two DOT codes
   cannot be the same airport.
4. The result is then **validated independently**: after mapping, each October route's
   `DISTANCE` must equal what that IATA pair flies in the clean months.

In [ ]:
oct_f   = flights_raw.filter(F.col("MONTH") == 10)
clean_f = flights_raw.filter(F.col("MONTH") != 10)


def airport_signature(df, scale: float = 1.0):
    """Fingerprint every airport by the distances it flies, in BOTH roles.

    An airport must be described by its arrivals as well as its departures.
    Code 10666 appears exactly once in October -- as a destination only. Building
    the signature from origins alone leaves it unmapped, and the airport join then
    silently drops that flight: the exact failure this notebook exists to prevent.
    """
    as_origin = df.select(F.col("ORIGIN_AIRPORT").alias("code"), "DISTANCE")
    as_dest   = df.select(F.col("DESTINATION_AIRPORT").alias("code"), "DISTANCE")
    return (as_origin.unionByName(as_dest)
            .groupBy("code")
            .agg(F.collect_set("DISTANCE").alias("distances"),
                 (F.count("*") / F.lit(scale)).alias("volume")))


oct_sig = airport_signature(oct_f).select(
    F.col("code").alias("dot"),
    F.col("distances").alias("d_oct"),
    F.col("volume").alias("v_oct"))

clean_sig = airport_signature(clean_f, scale=11.0).select(
    F.col("code").alias("iata"),
    F.col("distances").alias("d_clean"),
    F.col("volume").alias("v_expected"))

print(f"DOT codes to resolve : {oct_sig.count()}")
print(f"Candidate IATA codes : {clean_sig.count()}")

In [ ]:
# 306 x 322 candidate pairs is trivial to score exhaustively.
scored = (oct_sig.crossJoin(clean_sig)
    .withColumn("containment", F.size(F.array_intersect("d_oct", "d_clean")) / F.size("d_oct"))
    .withColumn("vol_sim", F.least("v_oct", "v_expected") / F.greatest("v_oct", "v_expected"))
    .withColumn("score", 0.6 * F.col("containment") + 0.4 * F.col("vol_sim"))
    .select("dot", "iata", "score", "containment", "vol_sim"))

# Greedy one-to-one assignment, highest score first.
candidates = sorted(scored.collect(), key=lambda r: -r["score"])
used_dot, used_iata, pairs = set(), set(), []
for r in candidates:
    if r["dot"] in used_dot or r["iata"] in used_iata:
        continue
    used_dot.add(r["dot"]); used_iata.add(r["iata"])
    pairs.append((r["dot"], r["iata"], float(r["score"]),
                  float(r["containment"]), float(r["vol_sim"])))

assert len({p[1] for p in pairs}) == len(pairs), "assignment is not one-to-one"
print(f"Resolved {len(pairs)} DOT codes to unique IATA codes")

print(f"\n  {'DOT':<8}{'IATA':<6}{'score':>7}{'contain':>9}{'vol_sim':>9}")
for d, i, s, ct, v in sorted(pairs, key=lambda p: p[2])[:6]:
    print(f"  {d:<8}{i:<6}{s:>7.3f}{ct:>9.3f}{v:>9.3f}   <- lowest confidence")

In [ ]:
code_map = spark.createDataFrame(
    [(d, i) for d, i, *_ in pairs], "dot_code string, iata_code string"
).cache()
code_map.show(5)

### Independent validation of the mapping

The mapping was built from distance *sets*. Validate it against per-route distances,
which it did not directly optimise for: after substitution, every October route's
`DISTANCE` must match the same IATA pair in the clean months.

A ±1 mile tolerance is allowed — the source data rounds inconsistently (ORD→CLE appears
as both 315 and 316), which is a source-data quirk, not a mapping error.

In [ ]:
route_distance = (clean_f.groupBy("ORIGIN_AIRPORT", "DESTINATION_AIRPORT")
                         .agg(F.first("DISTANCE").alias("known_distance")))

oct_mapped = (oct_f
    .join(code_map.withColumnRenamed("iata_code", "o_iata"),
          F.col("ORIGIN_AIRPORT") == F.col("dot_code"), "left").drop("dot_code")
    .join(code_map.withColumnRenamed("iata_code", "d_iata"),
          F.col("DESTINATION_AIRPORT") == F.col("dot_code"), "left").drop("dot_code"))

validation = (oct_mapped
    .join(route_distance,
          (F.col("o_iata") == route_distance.ORIGIN_AIRPORT) &
          (F.col("d_iata") == route_distance.DESTINATION_AIRPORT), "left")
    .select("o_iata", "DISTANCE", "known_distance")
    .withColumn("ok", F.abs(F.col("DISTANCE") - F.col("known_distance")) <= 1))

total_oct = validation.count()
correct   = validation.filter("ok").count()
pct = 100 * correct / total_oct

print(f"October flights validated : {total_oct:,}")
print(f"Distance agrees (+/-1 mi) : {correct:,}  ({pct:.3f}%)")

assert pct > 99.0, f"mapping validation failed at {pct:.2f}% - do not proceed"
print("\nPASS - mapping accepted.")

In [ ]:
# Be honest about where it is weak: which airports validate poorly?
weak = (validation.groupBy("o_iata")
        .agg(F.count("*").alias("flights"),
             F.round(100 * F.sum(F.col("ok").cast("int")) / F.count("*"), 2).alias("pct_ok"))
        .filter((F.col("pct_ok") < 95) | F.col("pct_ok").isNull())
        .orderBy("pct_ok"))

n_weak_flights = weak.agg(F.sum("flights")).first()[0] or 0
print(f"Airports validating below 95%: {weak.count()}  "
      f"({n_weak_flights:,} flights = {100*n_weak_flights/N_RAW:.3f}% of the dataset)")
weak.show()
print("These are tiny/seasonal airports. Documented as a known limitation rather than hidden.")

In [ ]:
# Apply the mapping to the whole dataset. Non-October rows pass through unchanged.
flights = (flights_raw
    .join(code_map.withColumnRenamed("iata_code", "_o"),
          F.col("ORIGIN_AIRPORT") == F.col("dot_code"), "left").drop("dot_code")
    .join(code_map.withColumnRenamed("iata_code", "_d"),
          F.col("DESTINATION_AIRPORT") == F.col("dot_code"), "left").drop("dot_code")
    .withColumn("origin",      F.coalesce("_o", "ORIGIN_AIRPORT"))
    .withColumn("destination", F.coalesce("_d", "DESTINATION_AIRPORT"))
    .drop("_o", "_d"))

contract.check(flights, "1. airport codes normalised", expect_delta=0)

bad_o = flights.filter(F.length("origin") != 3).count()
bad_d = flights.filter(F.length("destination") != 3).count()
print(f"\nRows still holding a non-IATA origin      : {bad_o:,}")
print(f"Rows still holding a non-IATA destination : {bad_d:,}")
assert bad_o == 0 and bad_d == 0, "some airport codes were not resolved"

---
## 4. Rule 5 — deduplicate

Notebook 01 found 0 exact duplicates and exactly 1 business-key collision. The dedupe is
cheap and makes the notebook idempotent — re-running can never double-count.

In [ ]:
KEY = ["YEAR","MONTH","DAY","AIRLINE","FLIGHT_NUMBER","origin","SCHEDULED_DEPARTURE"]

before = flights.count()
flights = flights.dropDuplicates(KEY)
contract.check(flights, "5. deduplicated on business key", expect_delta=-(before - flights.count()), tolerance=0)

---
## 5. Rules 2, 3, 4, 11 — flight outcome, and structural nulls

Notebook 01 proved the nulls encode facts:

- 89,884 cancelled and 15,187 diverted flights have no arrival data — **by definition**.
- Delay-cause columns are populated for exactly the 1,063,439 flights arriving 15+ min late.
- 3,731 flights were cancelled **after** pushback, so `CANCELLED = 1` does not mean
  "never left the gate".

Rather than filling these, we make the reason explicit in a `status` column. Downstream
notebooks filter on `status` instead of guessing what a null means.

In [ ]:
flights = flights.withColumn(
    "status",
    F.when(F.col("CANCELLED") == 1, "cancelled")
     .when(F.col("DIVERTED")  == 1, "diverted")
     .otherwise("completed"))

# Rule 11: distinguish cancelled-at-gate from cancelled-after-pushback.
flights = flights.withColumn(
    "cancelled_after_pushback",
    (F.col("CANCELLED") == 1) & F.col("DEPARTURE_DELAY").isNotNull())

flights.groupBy("status").agg(
    F.count("*").alias("flights"),
    F.count(F.when(F.col("cancelled_after_pushback"), 1)).alias("after_pushback"),
    F.count("DEPARTURE_DELAY").alias("has_dep_delay"),
    F.count("ARRIVAL_DELAY").alias("has_arr_delay"),
).show()

In [ ]:
# Rule 4: decode the cancellation reason. NULL stays NULL for non-cancelled flights.
flights = flights.withColumn("cancellation_reason",
    F.when(F.col("CANCELLATION_REASON") == "A", "Carrier")
     .when(F.col("CANCELLATION_REASON") == "B", "Weather")
     .when(F.col("CANCELLATION_REASON") == "C", "National Air System")
     .when(F.col("CANCELLATION_REASON") == "D", "Security"))

flights.filter(F.col("status") == "cancelled").groupBy("cancellation_reason").count() \
       .orderBy(F.desc("count")).show()

In [ ]:
# Rule 3: delay causes are meaningful ONLY for flights that arrived 15+ min late.
# Fill with 0 inside that subset; leave NULL outside it so averages cannot be polluted.
CAUSES = {"AIR_SYSTEM_DELAY":"delay_nas", "SECURITY_DELAY":"delay_security",
          "AIRLINE_DELAY":"delay_carrier", "LATE_AIRCRAFT_DELAY":"delay_late_aircraft",
          "WEATHER_DELAY":"delay_weather"}

is_late = (F.col("status") == "completed") & (F.col("ARRIVAL_DELAY") >= 15)
for src, dst in CAUSES.items():
    flights = flights.withColumn(dst, F.when(is_late, F.coalesce(F.col(src), F.lit(0))))

contract.check(flights, "2-4,11. outcomes + causes", expect_delta=0)

---
## 6. Rule 7 — real timestamps from `HHMM` integers

`SCHEDULED_DEPARTURE` is `1435`, meaning 14:35 — not 1,435 of anything. Arithmetic on the
raw value is silently wrong (`1500 - 1435 = 65`, but the real gap is 25 minutes).

`2400` is a legal value meaning midnight, which also rolls the date forward.

In [ ]:
def hhmm_to_minutes(colname):
    """HHMM integer -> minutes since midnight. 2400 -> 0."""
    v = F.col(colname)
    return F.when(v.isNull(), None).otherwise(
        F.when(v == 2400, F.lit(0)).otherwise((v / 100).cast("int") * 60 + (v % 100)))

for src, dst in [("SCHEDULED_DEPARTURE","sched_dep_min"), ("DEPARTURE_TIME","actual_dep_min"),
                 ("SCHEDULED_ARRIVAL","sched_arr_min"),  ("ARRIVAL_TIME","actual_arr_min")]:
    flights = flights.withColumn(dst, hhmm_to_minutes(src))

flights = flights.withColumn("flight_date",
    F.make_date(F.col("YEAR"), F.col("MONTH"), F.col("DAY")))

flights.select("SCHEDULED_DEPARTURE","sched_dep_min","SCHEDULED_ARRIVAL","sched_arr_min",
               "flight_date").show(5)

In [ ]:
# Sanity: minutes must land in [0, 1439] and the date must be valid 2015.
bad_min = flights.filter((F.col("sched_dep_min") < 0) | (F.col("sched_dep_min") > 1439)).count()
bad_dt  = flights.filter(F.col("flight_date").isNull()).count()
print(f"Out-of-range minutes : {bad_min:,}")
print(f"Unparseable dates    : {bad_dt:,}")
assert bad_min == 0 and bad_dt == 0
contract.check(flights, "7. times converted", expect_delta=0)

---
## 7. Rules 6, 8 — analysis dimensions and the ML target

Negative delays are kept: 57.2% of flights depart early and that is real signal.

`is_delayed` uses the standard DOT threshold of **15 minutes on arrival**, and is defined
only for completed flights — a cancelled flight is neither delayed nor on time, and
counting it as "not delayed" would flatter every airline with a high cancellation rate.

In [ ]:
completed = F.col("status") == "completed"

flights = (flights
    .withColumn("is_delayed", F.when(completed, (F.col("ARRIVAL_DELAY") >= 15).cast("int")))
    .withColumn("is_delayed_dep", F.when(completed, (F.col("DEPARTURE_DELAY") >= 15).cast("int")))
    .withColumn("delay_category",
        F.when(~completed, None)
         .when(F.col("ARRIVAL_DELAY") < 0,   "early")
         .when(F.col("ARRIVAL_DELAY") < 15,  "on_time")
         .when(F.col("ARRIVAL_DELAY") < 30,  "15-30 min")
         .when(F.col("ARRIVAL_DELAY") < 60,  "30-60 min")
         .when(F.col("ARRIVAL_DELAY") < 120, "1-2 hours")
         .otherwise("2+ hours"))
    .withColumn("sched_dep_hour", (F.col("sched_dep_min") / 60).cast("int"))
    .withColumn("time_of_day",
        F.when(F.col("sched_dep_hour") < 6,  "night")
         .when(F.col("sched_dep_hour") < 12, "morning")
         .when(F.col("sched_dep_hour") < 18, "afternoon")
         .otherwise("evening"))
    .withColumn("season",
        F.when(F.col("MONTH").isin(12, 1, 2), "winter")
         .when(F.col("MONTH").isin(3, 4, 5),  "spring")
         .when(F.col("MONTH").isin(6, 7, 8),  "summer")
         .otherwise("autumn"))
    .withColumn("is_weekend", F.col("DAY_OF_WEEK").isin(6, 7))
    .withColumn("route", F.concat_ws("-", "origin", "destination")))

contract.check(flights, "6,8. dimensions + ML target", expect_delta=0)

In [ ]:
flights.groupBy("delay_category").count().orderBy(F.desc("count")).show()

rate = flights.filter(completed).agg(F.round(100 * F.avg("is_delayed"), 2)).first()[0]
print(f"Arrival delay rate (completed flights only): {rate}%")
print(f"=> baseline 'never delayed' model accuracy : {100 - rate:.2f}%")

---
## 8. Rules 9, 10 — enrich with airline and airport metadata

Both joins are **LEFT** joins followed by an assertion that nothing failed to match.
This is exactly the failure notebook 01 found: an inner join here would have silently
deleted October.

In [ ]:
airlines = airlines_raw.select(
    F.col("IATA_CODE").alias("AIRLINE"), F.col("AIRLINE").alias("airline_name"))

flights = flights.join(F.broadcast(airlines), "AIRLINE", "left")
unmatched = flights.filter(F.col("airline_name").isNull()).count()
print(f"Flights with no airline name: {unmatched:,}")
assert unmatched == 0, "airline join lost rows"
contract.check(flights, "9. airline names joined", expect_delta=0)

In [ ]:
ap = airports_raw.select(
    F.col("IATA_CODE").alias("_code"),
    F.col("AIRPORT").alias("_name"), F.col("CITY").alias("_city"),
    F.col("STATE").alias("_state"),
    F.col("LATITUDE").alias("_lat"), F.col("LONGITUDE").alias("_lon"))

for side, prefix in [("origin", "origin"), ("destination", "dest")]:
    cols = {f"_{s}": f"{prefix}_{s}" for s in ["name","city","state","lat","lon"]}
    side_df = ap
    for old, new in cols.items():
        side_df = side_df.withColumnRenamed(old, new)
    flights = flights.join(F.broadcast(side_df), F.col(side) == F.col("_code"), "left").drop("_code")

miss_o = flights.filter(F.col("origin_name").isNull()).count()
miss_d = flights.filter(F.col("dest_name").isNull()).count()
print(f"Flights with unresolved origin airport      : {miss_o:,}")
print(f"Flights with unresolved destination airport : {miss_d:,}")
assert miss_o == 0 and miss_d == 0, "airport join lost rows -- the October bug is back"
contract.check(flights, "10. airport metadata joined", expect_delta=0)

In [ ]:
# Rule 10: 3 airports (ECP, PBG, UST) have no coordinates. They keep every metric;
# only the dashboard map skips them.
no_coords = flights.filter(F.col("origin_lat").isNull()).select("origin").distinct()
n_rows = flights.filter(F.col("origin_lat").isNull()).count()
print(f"Airports without coordinates: {[r[0] for r in no_coords.collect()]}")
print(f"Flights affected: {n_rows:,} ({100*n_rows/N_RAW:.3f}%) - retained in all metrics, "
      f"excluded from the map only.")

---
## 9. Final shape and write

Drop the raw `HHMM` and now-redundant columns, keep clean names, write partitioned Parquet.

In [ ]:
curated = flights.select(
    # identity
    "flight_date", F.col("YEAR").alias("year"), F.col("MONTH").alias("month"),
    F.col("DAY").alias("day"), F.col("DAY_OF_WEEK").alias("day_of_week"),
    F.col("AIRLINE").alias("airline_code"), "airline_name",
    F.col("FLIGHT_NUMBER").alias("flight_number"), F.col("TAIL_NUMBER").alias("tail_number"),
    # geography
    "origin", "origin_name", "origin_city", "origin_state", "origin_lat", "origin_lon",
    "destination", "dest_name", "dest_city", "dest_state", "dest_lat", "dest_lon", "route",
    # schedule
    "sched_dep_min", "sched_dep_hour", "actual_dep_min", "sched_arr_min", "actual_arr_min",
    F.col("SCHEDULED_TIME").alias("sched_duration"), F.col("ELAPSED_TIME").alias("actual_duration"),
    F.col("AIR_TIME").alias("air_time"), F.col("DISTANCE").alias("distance"),
    F.col("TAXI_OUT").alias("taxi_out"), F.col("TAXI_IN").alias("taxi_in"),
    # outcome
    "status", "cancelled_after_pushback", "cancellation_reason",
    F.col("DEPARTURE_DELAY").alias("dep_delay"), F.col("ARRIVAL_DELAY").alias("arr_delay"),
    "is_delayed", "is_delayed_dep", "delay_category",
    # causes
    "delay_carrier", "delay_weather", "delay_nas", "delay_security", "delay_late_aircraft",
    # dimensions
    "time_of_day", "season", "is_weekend",
)

print(f"Curated columns: {len(curated.columns)}  (raw was {len(flights_raw.columns)})")
curated.printSchema()

In [ ]:
t0 = time.time()
out = PATHS["curated"] / "flights.parquet"
curated.write.mode("overwrite").partitionBy("month").parquet(str(out))
print(f"Written in {time.time()-t0:.1f}s -> {out}")

In [ ]:
# Read it back and re-assert the contract survived the round trip.
check = spark.read.parquet(str(out))
n_final = check.count()
print(f"Rows written : {n_final:,}")
print(f"Rows raw     : {N_RAW:,}")
print(f"Difference   : {n_final - N_RAW:+,}")
assert n_final == contract.n, "row count changed on write"

import subprocess
size = subprocess.run(["du","-sh",str(out)], capture_output=True, text=True).stdout.split()[0]
print(f"On disk      : {size}")

---
## 10. Data contract summary

Final proof that the pipeline preserved every row it should have, and that the
October defect is genuinely fixed rather than hidden.

In [ ]:
print(f"{'STAGE':<34}{'ROWS':>12}{'DELTA':>12}")
print("-" * 58)
for label, n, d in contract.log:
    print(f"{label:<34}{n:>12,}{'' if d == 0 else f'{d:+,}':>12}")
print("-" * 58)
print(f"{'RETAINED':<34}{100*contract.n/N_RAW:>11.3f}%")

In [ ]:
# The decisive test: October must now behave like every other month.
(check.groupBy("month")
      .agg(F.count("*").alias("flights"),
           F.countDistinct("origin").alias("airports"),
           F.count(F.when(F.col("origin_name").isNotNull(), 1)).alias("with_metadata"),
           F.round(100 * F.avg("is_delayed"), 2).alias("delay_rate_pct"))
      .orderBy("month").show(12))

print("If October's 'with_metadata' equals its 'flights', the join defect is fixed.")

In [ ]:
spark.stop()
print("Notebook 02 complete. Curated dataset ready for notebooks 03-09.")